# Download and split HOB-CNNv2 combined images

This notebook downloads image files from the `data_combined` folder of the
[HOB-CNNv2 GitHub repository](https://github.com/ZijueChen/HOBCNN-v2), then splits
each combined image into four crops:

- `Summer_RGB`
- `Summer_GT`
- `Winter_RGB`
- `Winter_GT`

The output is saved inside your **Major Project** folder at:

`C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\HOB-CNN Images Split`

**Important:** the crop arrangement is based on the example you supplied:
top-left = Winter RGB, top-right = Summer RGB, bottom-left = Winter GT,
bottom-right = Summer GT. Check several outputs before using them. If the repository
uses a different layout, change `CROP_LAYOUT` below.

The notebook does not train HOB-CNNv2 and does not overwrite existing crops by default.


In [ ]:
# If needed, install the required packages in this notebook's environment.
%pip install requests pillow


In [ ]:
from pathlib import Path
from urllib.parse import quote
import io
import sys
import time

import requests
from PIL import Image, UnidentifiedImageError

REPO_OWNER = "ZijueChen"
REPO_NAME = "HOBCNN-v2"
DATA_FOLDER = "data_combined"
GITHUB_API = "https://api.github.com"
RAW_BASE = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}"

# Change this only if your Windows username or folder structure differs.
OUTPUT_ROOT = Path(
    r"C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3"
    r"\Numerical Methods and Machine Learning\Major Project\HOB-CNN Images Split"
)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

# Positions in the 2x2 combined image, inferred from the example supplied.
# (column, row): top-left=(0,0), top-right=(1,0), bottom-left=(0,1), bottom-right=(1,1)
CROP_LAYOUT = {
    "Winter_RGB": (0, 0),
    "Summer_RGB": (1, 0),
    "Winter_GT": (0, 1),
    "Summer_GT": (1, 1),
}

OVERWRITE_EXISTING = False

for folder_name in CROP_LAYOUT:
    (OUTPUT_ROOT / folder_name).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_ROOT)
print("Folders ready:", ", ".join(CROP_LAYOUT))


In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "HOB-CNNv2-dataset-splitter/1.0",
    "Accept": "application/vnd.github+json",
})

def get_json(url):
    response = session.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

def get_default_branch():
    repo_info = get_json(f"{GITHUB_API}/repos/{REPO_OWNER}/{REPO_NAME}")
    return repo_info["default_branch"]

def list_combined_images(branch):
    tree_url = (
        f"{GITHUB_API}/repos/{REPO_OWNER}/{REPO_NAME}/git/trees/"
        f"{quote(branch, safe='')}?recursive=1"
    )
    tree_data = get_json(tree_url)

    if tree_data.get("truncated"):
        raise RuntimeError(
            "GitHub returned a truncated repository tree. "
            "A repository ZIP download may be needed instead."
        )

    prefix = DATA_FOLDER.rstrip("/") + "/"
    paths = []
    for item in tree_data.get("tree", []):
        repo_path = item.get("path", "")
        if (
            item.get("type") == "blob"
            and repo_path.startswith(prefix)
            and Path(repo_path).suffix.lower() in IMAGE_EXTENSIONS
        ):
            paths.append(repo_path)

    return sorted(paths)

branch = get_default_branch()
image_paths = list_combined_images(branch)

print("Default branch:", branch)
print("Image files found:", len(image_paths))
if not image_paths:
    print("No images found. Confirm that the repository contains data_combined/ and that the folder name is correct.")
else:
    print("First few paths:")
    for p in image_paths[:10]:
        print(" -", p)


In [ ]:
def download_image(repo_path, branch):
    url = f"{RAW_BASE}/{quote(branch, safe='')}/{quote(repo_path, safe='/')}"
    response = session.get(url, timeout=120)
    response.raise_for_status()
    image = Image.open(io.BytesIO(response.content))
    image.load()
    return image

def split_into_quadrants(image):
    # Preserve RGB/RGBA/greyscale pixel values as much as possible.
    width, height = image.size
    if width < 2 or height < 2:
        raise ValueError(f"Image is too small to split: {width}x{height}")

    mid_x = width // 2
    mid_y = height // 2
    boxes = {
        (0, 0): (0, 0, mid_x, mid_y),
        (1, 0): (mid_x, 0, width, mid_y),
        (0, 1): (0, mid_y, mid_x, height),
        (1, 1): (mid_x, mid_y, width, height),
    }
    return {
        folder: image.crop(boxes[position])
        for folder, position in CROP_LAYOUT.items()
    }

print("Download and crop functions are ready.")


In [ ]:
# Run the download-and-split process.
# Existing outputs are skipped unless OVERWRITE_EXISTING = True above.

processed = 0
skipped = 0
failed = []

for index, repo_path in enumerate(image_paths, start=1):
    output_name = f"{Path(repo_path).stem}.png"
    destinations = {
        folder: OUTPUT_ROOT / folder / output_name
        for folder in CROP_LAYOUT
    }

    if not OVERWRITE_EXISTING and all(path.exists() for path in destinations.values()):
        skipped += 1
        continue

    try:
        combined = download_image(repo_path, branch)
        crops = split_into_quadrants(combined)

        if combined.width != combined.height:
            print(
                f"Warning: {repo_path} is {combined.width}x{combined.height}; "
                "splitting into equal quadrants anyway."
            )

        for folder, crop in crops.items():
            destination = destinations[folder]
            if destination.exists() and not OVERWRITE_EXISTING:
                continue
            crop.save(destination, format="PNG")

        processed += 1

    except (requests.RequestException, UnidentifiedImageError, OSError, ValueError) as exc:
        failed.append((repo_path, str(exc)))
        print(f"FAILED: {repo_path}: {exc}")

    if index % 25 == 0 or index == len(image_paths):
        print(f"Progress: {index}/{len(image_paths)} files checked")

    time.sleep(0.03)

print("\nFinished.")
print("Newly processed combined images:", processed)
print("Skipped (all four outputs already existed):", skipped)
print("Failed downloads/crops:", len(failed))
for folder in CROP_LAYOUT:
    count = len(list((OUTPUT_ROOT / folder).glob("*.png")))
    print(f"{folder}: {count} PNG files")

if failed:
    report_path = OUTPUT_ROOT / "failed_downloads.txt"
    report_path.write_text(
        "\n".join(f"{path}\t{error}" for path, error in failed),
        encoding="utf-8",
    )
    print("Failure details written to:", report_path)

print("\nReminder: inspect several crops in each folder to verify the quadrant assignments.")


## Quick output check

The cell below lists the first few files in each folder. Open a few images from every folder to verify that the crop order matches the dataset before using it for model evaluation.


In [ ]:
for folder in CROP_LAYOUT:
    files = sorted((OUTPUT_ROOT / folder).glob("*.png"))
    print(f"\\n{folder} — {len(files)} images")
    for file in files[:5]:
        with Image.open(file) as im:
            print(f"  {file.name}: {im.size} pixels, mode={im.mode}")


In [ ]:
import re
from pathlib import Path

def get_tree_ids(folder):
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    ids = set()

    for file in folder.iterdir():
        if not file.is_file() or file.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        name = file.stem.upper()

        # Remove seasonal / image-type suffixes
        name = re.sub(
            r"_(SUMMER|WINTER)_(RGB|GT|D)$",
            "",
            name
        )

        ids.add(name)

    return ids


# Only compare Summer RGB images
hob_names = get_tree_ids(hob_summer_rgb)
corrected_names = get_tree_ids(corrected_summer_rgb)

overlapping = hob_names & corrected_names
hob_only = hob_names - corrected_names
corrected_only = corrected_names - hob_names

print("=" * 55)
print("SUMMER RGB FILENAME COMPARISON")
print("=" * 55)

print(f"HOB-CNNv2 Summer RGB images: {len(hob_names)}")
print(f"Corrected Summer RGB images: {len(corrected_names)}")
print(f"Overlapping tree IDs: {len(overlapping)}")

print("\nOverlapping IDs:")
for name in sorted(overlapping):
    print(name)

print(f"\nHOB-CNNv2-only IDs: {len(hob_only)}")
print(f"Corrected-Summer-only IDs: {len(corrected_only)}")